# TPC Ablation Study - Google Colab

**IMPORTANT: Reset runtime first if rerunning**
- Runtime → Disconnect and delete runtime (to avoid nested folders)

**Steps:**
1. Enable GPU: Runtime → Change runtime type → T4 GPU → Save
2. Run cells 1-3 to setup (clone, install, mount Drive)
3. Run cell 4 to download MIMIC-IV data (~15-20 min, 9.92 GB)
4. Verify cell 4 shows '✓ READY!' before continuing
5. Run cell 5 to configure paths
6. Run cell 6 to start training (2-4 hours)
7. Results save to MyDrive/tpc_ablation_results/

In [ ]:
# Clone repository (only if not already cloned)
import os
if not os.path.exists('/content/PyHealth'):
    !git clone https://github.com/tarakjc2c/PyHealth.git
    print('✓ Cloned repository')
else:
    print('✓ Repository already exists')

%cd /content/PyHealth
!git checkout pr-1028

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install litdata polars pandas dask mne rdkit peft transformers ogb -q
import os
print(f'✓ Installed in: {os.getcwd()}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Download MIMIC-IV data from shared Google Drive
!pip install gdown -q
import os
import shutil

MIMIC_ROOT = '/content/mimic-iv'
GDRIVE_FOLDER_ID = '15vyfKQ6H0g7DVEbI8vAMNhr4fi2lznVn'

# Check if data already exists
if os.path.exists(f'{MIMIC_ROOT}/hosp/diagnoses_icd.csv.gz'):
    print('✓ MIMIC-IV data already exists')
else:
    print('Downloading MIMIC-IV data (9.92 GB, ~15-20 minutes)...')
    print('Downloading 34 files from shared Google Drive\n')
    
    # Download to temporary location
    temp_dir = '/content/mimic-download'
    !gdown --folder {GDRIVE_FOLDER_ID} -O {temp_dir} --remaining-ok
    
    print('\nOrganizing files...')
    # Find where gdown put the files
    if os.path.exists(temp_dir):
        !ls -la {temp_dir}
        
        # Check for nested folder structure
        contents = os.listdir(temp_dir)
        print(f'Downloaded contents: {contents}')
        
        # Look for hosp and icu folders
        found = False
        for item in contents:
            item_path = os.path.join(temp_dir, item)
            if os.path.isdir(item_path):
                # Check if this folder contains hosp/icu
                if os.path.exists(f'{item_path}/hosp') and os.path.exists(f'{item_path}/icu'):
                    shutil.move(item_path, MIMIC_ROOT)
                    found = True
                    break
        
        # If hosp/icu are directly in temp_dir
        if not found and 'hosp' in contents and 'icu' in contents:
            shutil.move(temp_dir, MIMIC_ROOT)
            found = True
        
        if found:
            print(f'✓ Data organized at {MIMIC_ROOT}')
        else:
            print('✗ ERROR: Could not find hosp/icu folders in download')
            print('Manual check needed - run: !find /content -name "chartevents.csv.gz" -type f')
    else:
        print('✗ ERROR: Download directory not created')

# Verify critical files exist
print('\nVerifying data files:')
critical_files = [
    'hosp/diagnoses_icd.csv.gz',
    'hosp/patients.csv.gz',
    'icu/chartevents.csv.gz',
    'icu/icustays.csv.gz'
]

all_found = True
for f in critical_files:
    path = os.path.join(MIMIC_ROOT, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f'✓ {f} ({size_mb:.1f} MB)')
    else:
        print(f'✗ MISSING: {f}')
        all_found = False

if all_found:
    hosp_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/hosp') if f.endswith('.csv.gz')])
    icu_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/icu') if f.endswith('.csv.gz')])
    print(f'\n✓ READY! Found {hosp_files} hosp files, {icu_files} icu files')
else:
    print('\n✗ Download incomplete. Try running this cell again.')

In [ ]:
# Fix paths for Colab
script = 'examples/length_of_stay/length_of_stay_mimic4_tpc.py'

with open(script, 'r') as f:
    lines = f.readlines()

# Replace path definitions
new_lines = []
for line in lines:
    if 'MIMIC_ROOT = r"C:' in line:
        new_lines.append('MIMIC_ROOT = "/content/mimic-iv"\n')
    elif 'CACHE_PATH = r"C:' in line:
        new_lines.append('CACHE_PATH = "/content/tpc_cache"\n')
    elif 'OUTPUT_DIR = "tpc_ablation_results"' in line:
        new_lines.append('OUTPUT_DIR = "/content/drive/MyDrive/tpc_ablation_results"\n')
    else:
        new_lines.append(line)

with open(script, 'w') as f:
    f.writelines(new_lines)

print('✓ Paths configured for Colab')

In [ ]:
# Run ablation study (2-4 hours)
!python examples/length_of_stay/length_of_stay_mimic4_tpc.py

## Download Results

Results are in MyDrive/tpc_ablation_results/
Download them with the cell below:

In [ ]:
from google.colab import files
import os

result_dir = '/content/drive/MyDrive/tpc_ablation_results'
for filename in ['ablation_results.json', 'mc_dropout_results.json']:
    filepath = os.path.join(result_dir, filename)
    if os.path.exists(filepath):
        print(f'Downloading: {filename}')
        files.download(filepath)